# Optimización de hiperparámetros de Random Forest
**Optimización del portafolio de variedades de rosa — Bryan Steve Vargas Cabascango**

Búsqueda en rejilla con **validación cruzada temporal de ventana expansiva** sobre el período de
entrenamiento (anterior a enero de 2025). El período de prueba no interviene en la búsqueda.

Se ejecuta **después** de `02_Modelado_Predictivo_Prescriptivo.ipynb`, porque la última sección
compara la asignación de superficie obtenida con la configuración ajustada frente a la de producción.

| Hiperparámetro | Valores explorados |
|---|---|
| `max_features` (fracción de variables por división) | 0,3 · 0,5 · 0,7 · 1,0 |
| `min_samples_leaf` (observaciones mínimas por hoja) | 1 · 5 · 10 · 20 |
| `max_depth` (profundidad máxima) | sin límite · 12 |

32 combinaciones × 4 pliegues. Durante la búsqueda se usan 150 árboles para acotar el tiempo;
las configuraciones finalistas se reentrenan con 500. **Tiempo aproximado: 6 minutos.**

In [ ]:
import warnings, os, json, itertools, time
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.covariance import LedoitWolf
from scipy.optimize import minimize

SALIDA = './modelo/'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'figure.dpi':110,
                     'axes.spines.top':False,'axes.spines.right':False})

d = pd.read_pickle('./figuras/panel_analitico.pkl').sort_values(['variety','id_month'])
g = d.groupby('variety', group_keys=False)
for L in (1,2,3,12): d[f'lag{L}'] = g.margin_m2.shift(L)
d['roll3']    = g.margin_m2.transform(lambda s: s.shift(1).rolling(3).mean())
d['roll12']   = g.margin_m2.transform(lambda s: s.shift(1).rolling(12).mean())
d['exp_mean'] = g.margin_m2.transform(lambda s: s.shift(1).expanding().mean())
d['exp_std']  = g.margin_m2.transform(lambda s: s.shift(1).expanding().std())
d['lag_stems']=g.stems_m2.shift(1); d['lag_nac']=g.pct_nacional.shift(1)
d['lag_nocol']=g.pct_no_colocada.shift(1); d['lag_price']=g.avg_price.shift(1)
d['mes']=d.id_month.dt.month; d['trend']=(d.id_month.dt.year-2020)*12+d.mes
FEATS=['lag1','lag2','lag3','lag12','roll3','roll12','exp_mean','exp_std','lag_stems','lag_nac',
       'lag_nocol','lag_price','mes','trend','density','monthly_flowers_per_plant','area','n_plants',
       'is_valentin','is_mothers_day_season','is_all_saints_season','is_christmas_season','is_covid_shock']
dm = d.dropna(subset=['lag12','roll12','exp_mean','margin_m2']).copy(); dm['exp_std'] = dm.exp_std.fillna(0)
tr, te = dm[dm.id_month < '2025-01-01'], dm[dm.id_month >= '2025-01-01']
print(f'Entrenamiento: {len(tr):,} obs | Prueba: {len(te):,} obs')

## 1. Pliegues temporales

Cuatro bloques de validación de seis meses. En cada uno se entrena con **toda** la información
anterior al bloque, lo que reproduce la forma en que el modelo se usaría en la práctica.

In [ ]:
BLOQUES = [('2023-01-01','2023-07-01'), ('2023-07-01','2024-01-01'),
           ('2024-01-01','2024-07-01'), ('2024-07-01','2025-01-01')]
GRID = list(itertools.product([0.3, 0.5, 0.7, 1.0], [1, 5, 10, 20], [None, 12]))

t0 = time.time(); filas = []
for mf, msl, md_ in GRID:
    sc = []
    for a, b in BLOQUES:
        A = tr[tr.id_month < a]; B = tr[(tr.id_month >= a) & (tr.id_month < b)]
        m = RandomForestRegressor(n_estimators=150, max_features=mf, min_samples_leaf=msl,
                                  max_depth=md_, random_state=42, n_jobs=-1).fit(A[FEATS], A.margin_m2)
        sc.append(np.sqrt(mean_squared_error(B.margin_m2, m.predict(B[FEATS]))))
    filas.append(dict(max_features=mf, min_samples_leaf=msl,
                      max_depth='Sin límite' if md_ is None else md_,
                      rmse_cv=np.mean(sc), ee_cv=np.std(sc, ddof=1)/np.sqrt(len(sc))))
R = pd.DataFrame(filas).sort_values('rmse_cv').reset_index(drop=True)
R.to_csv(SALIDA + 'tuning_rf.csv', index=False)
print(f'Búsqueda completada en {time.time()-t0:.0f} s')
R.head(10).round(4)

## 2. Regla de un error estándar

Las diferencias entre configuraciones deben juzgarse contra la variabilidad entre pliegues. La
**regla de un error estándar** establece que cualquier configuración cuyo error medio no supere
al de la mejor más un error estándar es estadísticamente equivalente a ella.

In [ ]:
best = R.iloc[0]
umbral = best.rmse_cv + best.ee_cv
k = R.index[(R.max_features == 0.5) & (R.min_samples_leaf == 5) & (R.max_depth == 'Sin límite')][0]
usada = R.loc[k]
print(f'Mejor configuración:  max_features={best.max_features}, min_samples_leaf={int(best.min_samples_leaf)}, '
      f'max_depth={best.max_depth}  ->  RMSE CV {best.rmse_cv:.4f} (EE {best.ee_cv:.4f})')
print(f'Umbral de un error estándar: {umbral:.4f}')
print(f'Configuración de producción: puesto {k+1} de {len(R)}  ->  RMSE CV {usada.rmse_cv:.4f}  '
      f'({(usada.rmse_cv/best.rmse_cv-1)*100:.2f} % sobre la mejor)')
print(f'Configuraciones equivalentes a la mejor: {(R.rmse_cv <= umbral).sum()} de {len(R)}')
print('¿La configuración de producción es equivalente?', 'SÍ' if usada.rmse_cv <= umbral else 'NO')

In [ ]:
H = R[R.max_depth == 'Sin límite'].pivot(index='min_samples_leaf', columns='max_features', values='rmse_cv')
fig, ax = plt.subplots(figsize=(6.8, 4.2))
im = ax.imshow(H.values, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(H.shape[1])); ax.set_xticklabels([f'{x:.1f}' for x in H.columns])
ax.set_yticks(range(H.shape[0])); ax.set_yticklabels(H.index)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        tag = ' ★' if (H.index[i]==best.min_samples_leaf and H.columns[j]==best.max_features) else \
              (' ●' if (H.index[i]==5 and H.columns[j]==0.5) else '')
        ax.text(j, i, f'{H.values[i,j]:.4f}{tag}', ha='center', va='center', fontsize=9)
ax.set_xlabel('Proporción de variables por división (max_features)')
ax.set_ylabel('Observaciones mínimas por hoja')
ax.set_title('RMSE de validación cruzada temporal (★ mejor, ● producción)')
plt.colorbar(im, ax=ax, shrink=.8); fig.tight_layout()
fig.savefig(SALIDA + 'h01_tuning_rf.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Evaluación en el período de prueba (500 árboles)

In [ ]:
def evaluar(mf, msl, md_):
    m = RandomForestRegressor(n_estimators=500, max_features=mf, min_samples_leaf=msl,
                              max_depth=md_, random_state=42, n_jobs=-1).fit(tr[FEATS], tr.margin_m2)
    p = m.predict(te[FEATS])
    return dict(RMSE=np.sqrt(mean_squared_error(te.margin_m2, p)),
                MAE=mean_absolute_error(te.margin_m2, p), R2=r2_score(te.margin_m2, p))
md_best = None if best.max_depth == 'Sin límite' else int(best.max_depth)
ev = pd.DataFrame({'Producción (0,5 · 5 · sin límite)': evaluar(0.5, 5, None),
                   'Ajustada (mejor en CV)': evaluar(float(best.max_features), int(best.min_samples_leaf), md_best)}).T
base = np.sqrt(mean_squared_error(te.margin_m2, te.exp_mean))
ev['Mejora sobre media histórica %'] = (base - ev.RMSE) / base * 100
ev.round(4)

## 4. ¿Cambia la recomendación de superficie?

La pregunta relevante para el proyecto no es si mejora la predicción, sino si cambia la
decisión. Se repite la ventana expansiva con la configuración ajustada, se reestima la matriz
de riesgo y se vuelve a optimizar con las mismas restricciones.

In [ ]:
res = []
for y in range(2022, 2027):
    A = dm[dm.id_month.dt.year < y]; B = dm[dm.id_month.dt.year == y]
    m = RandomForestRegressor(n_estimators=500, max_features=float(best.max_features),
                              min_samples_leaf=int(best.min_samples_leaf), max_depth=md_best,
                              random_state=42, n_jobs=-1).fit(A[FEATS], A.margin_m2)
    p = m.predict(B[FEATS]); res.append(B[['variety','id_month','margin_m2']].assign(pred=p, resid=B.margin_m2.values-p))
Rt = pd.concat(res); Rw = Rt[Rt.id_month >= '2022-01-01']
M = Rw.pivot_table(index='id_month', columns='variety', values='resid'); M = M.loc[:, M.notna().mean() >= .95].fillna(0.)
V = list(M.columns); n = len(V); S = LedoitWolf().fit(M.values).covariance_ * 12
mu = Rw.pivot_table(index='id_month', columns='variety', values='pred')[V].tail(12).mean().values * 12
ar = d[d.id_month == d.id_month.max()].set_index('variety').area.reindex(V).fillna(0)
w0 = (ar/ar.sum()).values; AT = float(ar.sum()); s0 = np.sqrt(w0@S@w0)
ub = np.minimum(.25, np.maximum(w0*2, .005)); I_ = np.eye(n)
cons = [{'type':'eq','fun':lambda z: z[:n].sum()-1,'jac':lambda z: np.r_[np.ones(n),np.zeros(n)]},
        {'type':'ineq','fun':lambda z: z[n:]-(z[:n]-w0),'jac':lambda z: np.hstack([-I_,I_])},
        {'type':'ineq','fun':lambda z: z[n:]+(z[:n]-w0),'jac':lambda z: np.hstack([I_,I_])},
        {'type':'ineq','fun':lambda z: .4-z[n:].sum(),'jac':lambda z: np.r_[np.zeros(n),-np.ones(n)]},
        {'type':'ineq','fun':lambda z: s0**2-z[:n]@S@z[:n],'jac':lambda z: np.r_[-2*S@z[:n],np.zeros(n)]}]
for ftol in (1e-9, 1e-8, 1e-7):
    r = minimize(lambda z: -(mu@z[:n]), np.r_[w0,np.zeros(n)], jac=lambda z: np.r_[-mu,np.zeros(n)],
                 method='SLSQP', bounds=list(zip(np.zeros(n),ub))+[(0,1)]*n, constraints=cons,
                 options={'maxiter':2000,'ftol':ftol})
    if r.success: break
w_aj = pd.Series(r.x[:n], index=V)
A0 = pd.read_csv(SALIDA + 'asignacion.csv').set_index('variety')
com = A0.index.intersection(V)
el0 = set(A0[A0.share_optimo < 1e-2].index) & set(A0[A0.share_actual > 1e-2].index)
el1 = set(w_aj[w_aj < 1e-4].index) & set(A0[A0.share_actual > 1e-2].index)
impacto = dict(ganancia_produccion=float((A0.area_optima_m2*A0.margen_esperado_m2_anio).sum()/AT -
                                         (A0.area_actual_m2*A0.margen_esperado_m2_anio).sum()/AT) * AT,
               ganancia_ajustada=float(mu@w_aj.values - mu@w0) * AT,
               corr_pesos=float(np.corrcoef(A0.loc[com,'share_optimo'], w_aj[com])[0,1]),
               eliminadas_produccion=len(el0), eliminadas_ajustada=len(el1), eliminadas_comunes=len(el0 & el1))
print(json.dumps(impacto, indent=1))
json.dump(dict(mejor=best.to_dict(), puesto_produccion=int(k+1), evaluacion=ev.to_dict(), impacto=impacto),
          open(SALIDA + 'tuning_resumen.json', 'w'), indent=1, default=str, ensure_ascii=False)

## Conclusión

Si la configuración de producción es estadísticamente equivalente a la mejor según la regla de
un error estándar, y la asignación resultante con la configuración ajustada es prácticamente la
misma, la recomendación del proyecto es **invariante a los hiperparámetros** y se conserva la
configuración de producción, que es la que sustenta todos los resultados reportados. La
configuración ajustada queda como recomendada para la reestimación anual.